In [1]:
import os

In [2]:
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\wine-quality\\notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\SAAD TARIQ\\github_repositories\\wine-quality'

In [5]:
import urllib.request as request
import zipfile
from dataclasses import dataclass
from pathlib import Path
from src.wine_quality_prediction.constants import *
from src.wine_quality_prediction.utils.common import read_yaml, create_directories
from src.wine_quality_prediction import logger

In [6]:
@dataclass
class DataIngestionConfig:
    root_directory: Path
    source_URL: str
    local_data_file: Path
    unzip_directory: Path

In [7]:
class ConfigurationManager:
    def __init__(self, config_file_path=CONFIG_FILE_PATH,
                 params_file_path=PARAMS_FILE_PATH,
                 schema_file_path=SCHEMA_FILE_PATH):
        self.config_file_path = read_yaml(path_to_yaml=config_file_path)
        self.params_file_path = read_yaml(path_to_yaml=params_file_path)
        self.schema_file_path = read_yaml(path_to_yaml=schema_file_path)

        create_directories([self.config_file_path.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config_file_path.data_ingestion

        create_directories([config.root_directory])

        data_ingestion_config = DataIngestionConfig(
            root_directory=config.root_directory,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_directory=config.unzip_directory
        )

        return data_ingestion_config

In [8]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_data(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"Data downloaded successfully and saved to {filename}")
        else:
            logger.info(f"Data file already exists at {self.config.local_data_file}. Skipping download.")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_directory
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(file=self.config.local_data_file, mode='r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [9]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-04 21:51:07,211] INFO: common: YAML file 'config\config.yaml' read successfully.]
[2026-03-04 21:51:07,211] INFO: common: YAML file 'params.yaml' read successfully.]
[2026-03-04 21:51:07,212] INFO: common: YAML file 'schema.yaml' read successfully.]
[2026-03-04 21:51:07,213] INFO: common: Directory 'artifacts' created successfully or already exists.]
[2026-03-04 21:51:07,214] INFO: common: Directory 'artifacts/data_ingestion' created successfully or already exists.]
[2026-03-04 21:51:08,241] INFO: 249633177: Data downloaded successfully and saved to artifacts/data_ingestion/data.zip]
